# Eksperimen 21B (EXP21B): Balanced Hyperparameter Tuning & Tree Regularization
Eksekusi pipeline eksperimen EXP21B pada Google Colab dengan GPU T4/A100.

**Fokus Utama Remediasi & Investigasi:**
1. Menghapus kebocoran quasi-identifier `Zip` dan `Merchant City` dari pohon XGBoost.
2. Menerapkan *Strict Out-of-Sample Preprocessing* (fit hanya di Train Window).
3. Systematic Balanced Hyperparameter Tuning: `scale_pos_weight`, `max_depth`, `min_child_weight`, `colsample_bytree`.
4. Kalibrasi threshold dinamis optimal tau* pada Validation Set.
5. Studi ablasi komparatif komprehensif untuk penyusunan tabel Bab 4 Skripsi.


In [ ]:
# Setup environment & install PyTorch Geometric
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install -q xgboost scikit-learn pyyaml matplotlib pandas numpy


In [ ]:
# Pastikan dataset tersedia di /content/
import os
csv_path = '/content/credit_card_transactions-ibm_v2.csv'
if not os.path.exists(csv_path):
    print('Dataset tidak ditemukan di', csv_path)
    print('Silakan upload credit_card_transactions-ibm_v2.csv ke folder /content/')
else:
    print('Dataset terverifikasi di:', csv_path)


In [ ]:
%%writefile config.colab.yaml
data:
  transactions: "/content/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_window:
  reference_date: "2018-01-01"
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

features:
  mode: "clean"
  use_cyclical_time: true
  exclude_high_cardinality: true

sampling:
  method: "uniform"
  undersampling_ratio: 1.0

model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "mean"
  dropout: 0.2

training:
  batch_size: 1024
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  label_smoothing: 0.0
  grad_clip_norm: 5.0
  warmup_epochs: 0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 7
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
  log_step_interval: 1
  num_neighbors: [15, 10]

# ==============================================================================
# PILAR III: EKSPLORASI HYPERPARAMETER TUNING & REGULARISASI POHON (EXP21B)
# ==============================================================================
tuning:
  enabled: true
  mode: "grid" # Pilihan: "grid", "ablation", "focused"
  selection_metric: "val_f1"
  min_precision_constraint: 0.10
  
  grid:
    scale_pos_weight: [1.0, 10.0, 27.0, 50.0]
    max_depth: [3, 4, 5]
    min_child_weight: [5, 20, 50]
    colsample_bytree: [0.6, 0.8, 1.0]
    
  base_params:
    n_estimators: 100
    learning_rate: 0.05
    subsample: 0.8
    
  run_ablation_benchmarks: true
  ablation_scale_pos_weights: [1.0, 10.0, 27.0, 50.0, "full"]
  ablation_feature_subsets: true

classifier:
  type: "xgboost"
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"

output:
  model_dir: "/content/model/exp21b"
  result_dir: "/content/result/exp21b"



In [ ]:
%%writefile config.py
import yaml
from pathlib import Path

def load_config(config_path: str = "config.reference.yaml") -> dict:
    """
    Memuat konfigurasi YAML dari file yang ditentukan.
    """
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"File konfigurasi tidak ditemukan: {config_path}")
        
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    return config



In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler
from pathlib import Path

def load_and_split_raw_data(
    file_path: str, 
    start_year: int = 2016, 
    nrows: int = None,
    reference_date: str = '2018-01-01',
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60
) -> tuple:
    """
    Memuat dataset mentah IBM Credit Card Transactions, memfilter era modern (Year >= start_year),
    melakukan parsing datetime kronologis, dan memotongnya ke dalam 4 partisi temporal SHINE FRAUD II
    SEBELUM proses rekayasa fitur atau penskalaan (Strict Out-of-Sample Protocol).
    """
    print(f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})...")
    
    # 1. Pemuatan chunked efisien jika start_year ditentukan
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000
        
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            filtered_chunk = chunk[chunk['Year'] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break
                    
        if not chunks:
            raise ValueError(f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}")
            
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data mentah...")
    print("Parsing timestamps...")
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("Sorting chronologically...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("Cleaning Amount & Target Label...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    # 2. Pembagian Jendela Waktu Temporal (SHINE FRAUD II)
    print("\nMelakukan pembagian temporal window strictly chronologic...")
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)
    
    train_df = df[(df['datetime'] >= t0) & (df['datetime'] < t_val_start)].copy().reset_index(drop=True)
    val_df = df[(df['datetime'] >= t_val_start) & (df['datetime'] < t_gap_start)].copy().reset_index(drop=True)
    gap_df = df[(df['datetime'] >= t_gap_start) & (df['datetime'] < t_test_start)].copy().reset_index(drop=True)
    test_df = df[(df['datetime'] >= t_test_start) & (df['datetime'] < t_test_end)].copy().reset_index(drop=True)
    
    def compute_split_stats(name: str, split_df: pd.DataFrame, start_dt, end_dt):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0,
                "fraud_rate_pct": 0.0,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0
            }
        n_fraud = int(split_df['Is Fraud?'].sum())
        n_users = int(split_df['User'].nunique())
        n_merchants = int(split_df['Merchant Name'].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4),
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)
    
    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment (EXP21B Anti-Leakage & Balanced Tuning)",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats("Validation", val_df, t_val_start, t_gap_start),
            "embargo_gap": compute_split_stats("Embargo Gap", gap_df, t_gap_start, t_test_start),
            "test": compute_split_stats("Test", test_df, t_test_start, t_test_end),
            "total_window": compute_split_stats("Total Window", total_window_df, t0, t_test_end)
        }
    }
    
    return train_df, val_df, test_df, gap_df, stats_dict

def apply_strict_anti_leakage_preprocessing(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_df: pd.DataFrame = None,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True
) -> tuple:
    """
    Menerapkan Strict Anti-Leakage Preprocessing Protocol:
    1. StandardScaler Amount HANYA di-fit pada train_df, lalu mentransformasikan val_df dan test_df.
    2. Encoding kategorikal HANYA mempelajari kamus nilai unik dari train_df. Nilai baru pada val/test dipetakan ke 0 (<oov>).
    3. Mengeliminasi Zip dan Merchant City (mode: clean) untuk mencegah pohon keputusan menghafal ID kuasi.
    4. Menambahkan fitur waktu siklik (sin/cos jam & hari) yang tidak rentan memorization.
    """
    print(f"\n[STRICT ANTI-LEAKAGE] Menerapkan preprocessing (Mode: {feature_mode}, Cyclical Time: {use_cyclical_time})...")
    
    # Tentukan atribut kategorikal yang diizinkan
    if feature_mode == "clean":
        cat_columns = ['Use Chip', 'Merchant State', 'MCC', 'Errors?']
        print(f"  • Fitur High-Cardinality ('Zip', 'Merchant City') DIHAPUS dari input pohon!")
    elif feature_mode == "clean_basic":
        cat_columns = ['Use Chip', 'MCC', 'Errors?']
        print(f"  • Menggunakan fitur kategorikal minimal: {cat_columns}")
    elif feature_mode == "legacy_leaky":
        cat_columns = ['Use Chip', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?']
        print(f"  • [WARNING] Mode legacy_leaky: Termasuk 'Zip' dan 'Merchant City'.")
    else:
        raise ValueError(f"Feature mode '{feature_mode}' tidak dikenal. Pilih 'clean', 'clean_basic', atau 'legacy_leaky'.")

    # A. Standarisasi Fitur Numerik (Amount) — FIT HANYA DI TRAIN
    scaler_amount = StandardScaler()
    train_df['Amount'] = scaler_amount.fit_transform(train_df[['Amount']])
    val_df['Amount'] = scaler_amount.transform(val_df[['Amount']])
    test_df['Amount'] = scaler_amount.transform(test_df[['Amount']])
    if gap_df is not None and len(gap_df) > 0:
        gap_df['Amount'] = scaler_amount.transform(gap_df[['Amount']])
        
    print(f"  • Amount Scaler fit pada Train: mean={scaler_amount.mean_[0]:.4f}, std={scaler_amount.scale_[0]:.4f}")

    # B. Robust Categorical Encoding — DICTIONARY FIT HANYA DI TRAIN
    categorical_encoders = {}
    for col in cat_columns:
        train_df[col] = train_df[col].fillna('<missing>').astype(str)
        val_df[col] = val_df[col].fillna('<missing>').astype(str)
        test_df[col] = test_df[col].fillna('<missing>').astype(str)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].fillna('<missing>').astype(str)
            
        # Pelajari mapping unik dari TRAIN saja
        unique_train_vals = sorted(train_df[col].unique().tolist())
        # Index 0 dicadangkan untuk OOV (<unknown>), kategori train mulai dari index 1
        cat_to_id = {val: idx + 1 for idx, val in enumerate(unique_train_vals)}
        categorical_encoders[col] = cat_to_id
        
        # Transform dengan fallback OOV = 0
        train_df[col] = train_df[col].map(cat_to_id).fillna(0).astype(int)
        val_df[col] = val_df[col].map(cat_to_id).fillna(0).astype(int)
        test_df[col] = test_df[col].map(cat_to_id).fillna(0).astype(int)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].map(cat_to_id).fillna(0).astype(int)
            
        num_oov_val = int((val_df[col] == 0).sum())
        num_oov_test = int((test_df[col] == 0).sum())
        print(f"  • Kolom '{col}': {len(cat_to_id)} kategori train dipelajari (OOV di Val: {num_oov_val}, di Test: {num_oov_test})")

    # C. Fitur Temporal Siklik (Cyclical Time Features)
    feature_cols = ['Amount'] + cat_columns
    if use_cyclical_time:
        for split_df in [train_df, val_df, test_df]:
            hour_float = split_df['datetime'].dt.hour + split_df['datetime'].dt.minute / 60.0
            split_df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24.0).astype(np.float32)
            split_df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24.0).astype(np.float32)
            
            dow = split_df['datetime'].dt.dayofweek
            split_df['dow_sin'] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            split_df['dow_cos'] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)
            
        if gap_df is not None and len(gap_df) > 0:
            hour_float = gap_df['datetime'].dt.hour + gap_df['datetime'].dt.minute / 60.0
            gap_df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24.0).astype(np.float32)
            gap_df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24.0).astype(np.float32)
            dow = gap_df['datetime'].dt.dayofweek
            gap_df['dow_sin'] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            gap_df['dow_cos'] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)

        time_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
        feature_cols = feature_cols + time_cols
        print(f"  • Ditambahkan 4 fitur waktu siklik: {time_cols}")

    print(f"Total fitur transaksi bersih (feature_cols): {len(feature_cols)} fitur -> {feature_cols}")
    
    artifacts = {
        "scaler_amount": scaler_amount,
        "categorical_encoders": categorical_encoders,
        "feature_cols": feature_cols,
        "feature_mode": feature_mode
    }
    
    return train_df, val_df, test_df, gap_df, feature_cols, artifacts



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21b & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list, scalers_dict: dict = None) -> tuple:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi bersih (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi
      - merchant: Agregat rata-rata Amount & jumlah transaksi
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)

    Anti-Leakage Guard:
    Jika scalers_dict is None, scaler untuk User dan Merchant di-fit pada data saat ini (hanya untuk Train).
    Jika scalers_dict disediakan, scaler dari Train digunakan untuk me-transform User dan Merchant
    pada Validation dan Test tanpa membocorkan statistik masa depan.
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id').reset_index(drop=True)
    
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id').reset_index(drop=True)
    
    # Penskalaan fitur User & Merchant (Strict Anti-Leakage)
    if scalers_dict is None:
        scaler_user = StandardScaler()
        user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
        
        scaler_merchant = StandardScaler()
        merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        
        out_scalers = {
            'scaler_user': scaler_user,
            'scaler_merchant': scaler_merchant
        }
    else:
        scaler_user = scalers_dict['scaler_user']
        scaler_merchant = scalers_dict['scaler_merchant']
        user_feats_scaled = scaler_user.transform(user_agg[['user_avg_amount', 'user_tx_count']])
        merchant_feats_scaled = scaler_merchant.transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        out_scalers = scalers_dict
        
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    # 3. Fitur Transaksi & Label
    tx_features = torch.tensor(df[feature_cols].values, dtype=torch.float)
    tx_labels = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    
    data['transaction'].x = tx_features
    data['transaction'].y = tx_labels
    data['transaction'].num_nodes = num_transactions
    
    data['user'].x = user_features
    data['user'].num_nodes = num_users
    
    data['merchant'].x = merchant_features
    data['merchant'].num_nodes = num_merchants
    
    # 4. Membangun Edge Index Bipartit
    tx_indices = np.arange(num_transactions)
    user_src_indices = df['User'].map(user_mapping).values
    merchant_src_indices = df['Merchant Name'].map(merchant_mapping).values
    
    edge_user_to_tx = torch.tensor(np.array([user_src_indices, tx_indices]), dtype=torch.long)
    edge_tx_to_user = torch.tensor(np.array([tx_indices, user_src_indices]), dtype=torch.long)
    
    edge_merchant_to_tx = torch.tensor(np.array([merchant_src_indices, tx_indices]), dtype=torch.long)
    edge_tx_to_merchant = torch.tensor(np.array([tx_indices, merchant_src_indices]), dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_user_to_tx
    data['transaction', 'rev_buys', 'user'].edge_index = edge_tx_to_user
    
    data['merchant', 'sells', 'transaction'].edge_index = edge_merchant_to_tx
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_tx_to_merchant
    
    print(f"HeteroData selesai: {num_transactions} tx, {num_users} users, {num_merchants} merchants.")
    return data, out_scalers



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self, 
        hidden_channels: int, 
        out_channels: int, 
        num_layers: int, 
        data_metadata: tuple, 
        feature_dims: dict, 
        aggr: str = 'mean',
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear bawaan untuk monitoring val AUPRC selama pelatihan GNN
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal
        h_dict = {}
        
        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
            elif node_type in self.param_embeddings:
                num_nodes = num_nodes_dict[node_type]
                h_dict[node_type] = self.param_embeddings[node_type].expand(num_nodes, -1)
                
        # 2. GraphSAGE Message Passing
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            
            if 'transaction' in h_dict and h_dict['transaction'].size(0) > 0:
                # Terapkan BatchNorm jika batch_size > 1
                if h_dict['transaction'].size(0) > 1:
                    h_dict['transaction'] = self.bns[i](h_dict['transaction'])
                h_dict['transaction'] = F.relu(h_dict['transaction'])
                h_dict['transaction'] = self.dropout(h_dict['transaction'])
                
        tx_repr = h_dict['transaction']
        
        if return_embeddings:
            return tx_repr
            
        logits = self.classifier(tx_repr).squeeze(-1)
        probs = torch.sigmoid(logits)
        return probs

def manual_weighted_bce(probs, targets, pos_weight):
    """
    Class-weighted Binary Cross-Entropy loss yang stabil secara numerik.
    """
    eps = 1e-7
    probs = torch.clamp(probs, eps, 1.0 - eps)
    loss = - (pos_weight * targets * torch.log(probs) + (1.0 - targets) * torch.log(1.0 - probs))
    return loss.mean()

def manual_unweighted_bce(probs, targets):
    """
    Unweighted BCE loss untuk monitoring loss objektif alami.
    """
    eps = 1e-7
    probs = torch.clamp(probs, eps, 1.0 - eps)
    loss = - (targets * torch.log(probs) + (1.0 - targets) * torch.log(1.0 - probs))
    return loss.mean()



In [ ]:
%%writefile samplers.py
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

def calculate_degree_centrality(data: HeteroData, edge_type: tuple):
    """Menghitung degree centrality sederhana untuk edge type tertentu."""
    src, rel, dst = edge_type
    edge_index = data[edge_type].edge_index
    num_nodes = data[src].num_nodes
    degree = torch.zeros(num_nodes, dtype=torch.float)
    degree.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    return degree

def apply_uniform_sampling(data: HeteroData):
    """
    Baseline: Uniform Random Sampling.
    Memberikan probabilitas yang sama (1.0) untuk semua edge.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        data[edge_type].edge_weight = torch.ones(num_edges, dtype=torch.float)
    return data

def apply_topology_aware_sampling(data: HeteroData):
    """
    Topology-Aware Sampling (Inverse Log-Degree Centrality - v5 Stabilized).
    Menggunakan inverse sublinear log1p(degree) untuk memprioritaskan edge struktural
    tanpa disparitas bobot yang ekstrem.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree)
        log_deg = torch.log1p(deg_src[src_nodes])
        weights = 1.0 / (log_deg + 1e-5)
        min_val = weights.min()
        max_val = weights.max()
        if (max_val - min_val) > 1e-5:
            weights = (weights - min_val) / (max_val - min_val)
        else:
            weights = torch.zeros_like(weights)
        data[edge_type].edge_weight = weights + 0.1
    return data

def apply_importance_based_sampling(data: HeteroData):
    """
    Importance-Based Sampling (Log-Degree Centrality Proxy - v5 Stabilized).
    Menggunakan transformasi sublinear log1p untuk meredam varians sampling
    akibat super-hub merchant dan mencegah anomali ledakan False Positive di akhir training.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree)
        log_deg = torch.log1p(deg_src[src_nodes])
        min_val = log_deg.min()
        max_val = log_deg.max()
        if (max_val - min_val) > 1e-5:
            weights = (log_deg - min_val) / (max_val - min_val)
        else:
            weights = torch.zeros_like(log_deg)
        data[edge_type].edge_weight = weights + 0.1
    return data

def get_neighbor_loader(
    data: HeteroData, 
    num_neighbors: list, 
    batch_size: int, 
    shuffle: bool = True, 
    sampling_method: str = "uniform",
    input_nodes: torch.Tensor = None
) -> NeighborLoader:
    """
    Membangun PyG NeighborLoader dengan metode sampling yang dipilih.
    """
    if sampling_method == "uniform":
        data = apply_uniform_sampling(data)
    elif sampling_method == "topology":
        data = apply_topology_aware_sampling(data)
    elif sampling_method == "importance":
        data = apply_importance_based_sampling(data)
    else:
        raise ValueError(f"Metode sampling tidak dikenal: {sampling_method}. Pilih 'uniform', 'topology', atau 'importance'.")
        
    if input_nodes is None:
        input_nodes_arg = ('transaction', None)
    else:
        input_nodes_arg = ('transaction', input_nodes)
        
    loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        batch_size=batch_size,
        input_nodes=input_nodes_arg,
        shuffle=shuffle,
        num_workers=0
    )
    return loader



In [ ]:
%%writefile visualize_results.py
import os
import json
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import precision_recall_curve, roc_curve

def setup_plot_style():
    plt.rcParams.update({
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 13,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight'
    })

def plot_feature_importance(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Feature Importance XGBoost: Membedakan secara visual kontribusi
    fitur tabular bersih vs representasi graf 128-d GNN.
    """
    for method in methods:
        imp_path = result_dir / f"exp21b_{method}_feature_importance.json"
        if not imp_path.exists():
            continue
            
        with open(imp_path, 'r') as f:
            data = json.load(f)
            
        top_features = data.get('top_features', [])[:15]
        if not top_features:
            continue
            
        names = [item['feature_name'] for item in reversed(top_features)]
        gains = [item['gain'] for item in reversed(top_features)]
        colors = ['#1f77b4' if item['is_gnn'] else '#ff7f0e' for item in reversed(top_features)]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        bars = ax.barh(names, gains, color=colors, edgecolor='black', linewidth=0.7)
        
        ax.set_xlabel('Split Gain (Tingkat Kepentingan Keputusan)')
        ax.set_title(f'EXP21B: Feature Importance XGBoost ({method.upper()})\n'
                     f'Tabular: {data["pct_tabular_gain"]:.1f}% | GNN Embeddings: {data["pct_gnn_gain"]:.1f}%', fontweight='bold')
        ax.grid(axis='x', linestyle='--', alpha=0.6)
        
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#ff7f0e', edgecolor='black', label='Clean Tabular Feature'),
            Patch(facecolor='#1f77b4', edgecolor='black', label='GraphSAGE Representation')
        ]
        ax.legend(handles=legend_elements, loc='lower right')
        
        plt.tight_layout()
        save_path = output_dir / f"exp21b_{method}_feature_importance.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_spw_sensitivity(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Sensitivitas Class-Weight Damping (scale_pos_weight):
    Menampilkan kurva trade-off antara Recall dan Precision serta kontribusi GNN.
    """
    for method in methods:
        abl_path = result_dir / f"exp21b_{method}_ablation_summary.json"
        if not abl_path.exists():
            continue
            
        with open(abl_path, 'r') as f:
            data = json.load(f)
            
        spw_data = data.get('scale_pos_weight_ablation', [])
        if not spw_data:
            continue
            
        spw_vals = [item['scale_pos_weight'] for item in spw_data]
        f1_vals = [item['test_f1'] for item in spw_data]
        rec_vals = [item['test_recall'] * 100 for item in spw_data]
        prec_vals = [item['test_precision'] * 100 for item in spw_data]
        gnn_gain = [item.get('pct_gnn_gain', 0.0) for item in spw_data]
        
        x_indices = np.arange(len(spw_vals))
        labels = [f"{v:.0f}" if v < 100 else f"{v:.0f} (Full)" for v in spw_vals]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
        
        # Plot 1: Trade-off Metrik Klasifikasi
        ax1.plot(x_indices, f1_vals, marker='o', lw=2.5, color='#2ca02c', label='F1-Score')
        ax1.plot(x_indices, [r/100 for r in rec_vals], marker='s', lw=2.0, color='#1f77b4', linestyle='--', label='Recall')
        ax1.plot(x_indices, [p/100 for p in prec_vals], marker='^', lw=2.0, color='#d62728', linestyle='--', label='Precision')
        
        ax1.set_xticks(x_indices)
        ax1.set_xticklabels(labels)
        ax1.set_xlabel('scale_pos_weight')
        ax1.set_ylabel('Skor Evaluasi pada Test Set')
        ax1.set_title(f'EXP21B ({method.upper()}): Trade-Off Precision-Recall vs SPW', fontweight='bold')
        ax1.grid(True, linestyle=':', alpha=0.6)
        ax1.legend(loc='best')
        
        # Plot 2: Persentase Gain GNN
        ax2.bar(x_indices, gnn_gain, color='#9467bd', edgecolor='black', width=0.55)
        ax2.set_xticks(x_indices)
        ax2.set_xticklabels(labels)
        ax2.set_xlabel('scale_pos_weight')
        ax2.set_ylabel('Share Gain Keputusan GNN (%)')
        ax2.set_title('Kontribusi Representasi Graf GraphSAGE', fontweight='bold')
        ax2.grid(axis='y', linestyle=':', alpha=0.6)
        
        for idx, val in enumerate(gnn_gain):
            ax2.text(idx, val + 0.5, f"{val:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')
            
        plt.tight_layout()
        save_path = output_dir / f"exp21b_{method}_spw_sensitivity.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_tuning_heatmap(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Heatmap Grid Tuning: max_depth vs min_child_weight.
    """
    for method in methods:
        csv_path = result_dir / f"exp21b_{method}_tuning_results.csv"
        if not csv_path.exists():
            continue
            
        df = pd.read_csv(csv_path)
        if 'max_depth' not in df.columns or 'min_child_weight' not in df.columns or 'val_f1' not in df.columns:
            continue
            
        pivot = df.pivot_table(index='max_depth', columns='min_child_weight', values='val_f1', aggfunc='mean')
        
        fig, ax = plt.subplots(figsize=(7, 5.5))
        cax = ax.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
        
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_xticklabels([f"mcw={c}" for c in pivot.columns])
        ax.set_yticklabels([f"depth={r}" for r in pivot.index])
        
        ax.set_xlabel('min_child_weight (Regularisasi Daun)')
        ax.set_ylabel('max_depth (Kedalaman Pohon)')
        ax.set_title(f'EXP21B ({method.upper()}): Mean Validation F1 Heatmap', fontweight='bold')
        
        # Tambahkan anotasi teks nilai pada setiap sel
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                color = "white" if val > pivot.values.mean() else "black"
                ax.text(j, i, f"{val:.4f}", ha='center', va='center', color=color, fontweight='bold')
                
        fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        save_path = output_dir / f"exp21b_{method}_depth_mcw_heatmap.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_pr_and_roc(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Kurva Precision-Recall dan ROC pada Test Set masa depan.
    """
    fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(14, 6))
    colors = {'uniform': '#1f77b4', 'topology': '#2ca02c', 'importance': '#d62728'}
    
    found_any = False
    for method in methods:
        pred_path = result_dir / f"exp21b_{method}_predictions.npz"
        res_path = result_dir / f"exp21b_{method}_results.json"
        if not (pred_path.exists() and res_path.exists()):
            continue
            
        found_any = True
        npz = np.load(pred_path)
        test_probs = npz['test_probs']
        test_labels = npz['test_labels']
        
        with open(res_path, 'r') as f:
            res = json.load(f)
            
        auprc = res['metrics_threshold_independent']['auprc']
        roc_auc = res['metrics_threshold_independent']['roc_auc']
        
        prec, rec, _ = precision_recall_curve(test_labels, test_probs)
        ax_pr.plot(rec, prec, label=f"{method.capitalize()} (AUCPR = {auprc:.4f})", color=colors.get(method, 'blue'), lw=2.2)
        
        fpr, tpr, _ = roc_curve(test_labels, test_probs)
        ax_roc.plot(fpr, tpr, label=f"{method.capitalize()} (ROC-AUC = {roc_auc:.4f})", color=colors.get(method, 'blue'), lw=2.2)
        
    if not found_any:
        plt.close()
        return

    # Baseline kurva PR (Proporsi kelas minoritas)
    ax_pr.axhline(y=float(test_labels.mean()), color='grey', linestyle='--', label=f'Baseline ({test_labels.mean()*100:.2f}%)')
    ax_pr.set_xlabel('Recall (Sensitivitas Deteksi Fraud)')
    ax_pr.set_ylabel('Precision (Ketepatan Alarm)')
    ax_pr.set_title('Precision-Recall Curves (Test Set 60 Hari)', fontweight='bold')
    ax_pr.set_xlim([0.0, 1.02])
    ax_pr.set_ylim([0.0, 1.02])
    ax_pr.grid(True, linestyle=':', alpha=0.6)
    ax_pr.legend(loc='best')
    
    # Baseline kurva ROC (Random Classifier)
    ax_roc.plot([0, 1], [0, 1], color='grey', linestyle='--', label='Random Classifier (0.50)')
    ax_roc.set_xlabel('False Positive Rate')
    ax_roc.set_ylabel('True Positive Rate')
    ax_roc.set_title('Receiver Operating Characteristic (ROC)', fontweight='bold')
    ax_roc.set_xlim([0.0, 1.02])
    ax_roc.set_ylim([0.0, 1.02])
    ax_roc.grid(True, linestyle=':', alpha=0.6)
    ax_roc.legend(loc='best')
    
    plt.tight_layout()
    save_path = output_dir / "exp21b_pr_roc_curves.png"
    plt.savefig(save_path)
    plt.close()
    print(f"Chart tersimpan: {save_path}")

def plot_ablation_subsets(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Ablasi Input Fitur: Tabular Only vs GraphSAGE Only vs Hybrid Tuned.
    """
    for method in methods:
        abl_path = result_dir / f"exp21b_{method}_ablation_summary.json"
        if not abl_path.exists():
            continue
            
        with open(abl_path, 'r') as f:
            data = json.load(f)
            
        subset_data = data.get('feature_subset_ablation', [])
        if not subset_data:
            continue
            
        subsets = [item['subset_name'] for item in subset_data]
        f1_scores = [item['test_f1'] for item in subset_data]
        auprc_scores = [item['test_auprc'] for item in subset_data]
        recall_scores = [item['test_recall'] * 100 for item in subset_data]
        prec_scores = [item['test_precision'] * 100 for item in subset_data]
        
        x = np.arange(len(subsets))
        width = 0.35
        
        fig, ax = plt.subplots(figsize=(9, 5.5))
        rects1 = ax.bar(x - width/2, f1_scores, width, label='F1-Score', color='#1f77b4', edgecolor='black')
        rects2 = ax.bar(x + width/2, auprc_scores, width, label='AUCPR', color='#2ca02c', edgecolor='black')
        
        ax.set_ylabel('Skor Metrik')
        ax.set_title(f'EXP21B ({method.upper()}): Kontribusi Nilai Tambah Representasi Graf (Lift)', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(subsets)
        ax.legend(loc='upper left')
        ax.grid(axis='y', linestyle=':', alpha=0.6)
        
        # Anotasi angka di atas bar
        for rect in rects1:
            h = rect.get_height()
            ax.annotate(f'{h:.4f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')
        for rect in rects2:
            h = rect.get_height()
            ax.annotate(f'{h:.4f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')
                        
        plt.tight_layout()
        save_path = output_dir / f"exp21b_{method}_ablation_subsets.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def main():
    parser = argparse.ArgumentParser(description="Generasi Visualisasi Ilmiah EXP21B")
    parser.add_argument("--result-dir", type=str, default="result/exp21b", help="Direktori hasil output")
    parser.add_argument("--output-dir", type=str, default=None, help="Direktori penyimpanan plot")
    args = parser.parse_args()

    result_dir = Path(args.result_dir)
    output_dir = Path(args.output_dir) if args.output_dir is not None else result_dir / "plots"
    output_dir.mkdir(parents=True, exist_ok=True)

    setup_plot_style()
    methods = ["uniform", "topology", "importance"]

    print("=" * 70)
    print("  MEMULAI GENERASI GRAFIK & VISUALISASI ILMIAH EXP21B (300 DPI)")
    print("=" * 70)

    plot_feature_importance(result_dir, output_dir, methods)
    plot_spw_sensitivity(result_dir, output_dir, methods)
    plot_tuning_heatmap(result_dir, output_dir, methods)
    plot_pr_and_roc(result_dir, output_dir, methods)
    plot_ablation_subsets(result_dir, output_dir, methods)

    print("=" * 70)
    print("  Semua visualisasi berhasil digenerate di:", output_dir)
    print("=" * 70)

if __name__ == "__main__":
    main()



In [ ]:
%%writefile main.py
import argparse
import time
import json
import gc
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    accuracy_score, roc_auc_score, average_precision_score, 
    confusion_matrix
)
from xgboost import XGBClassifier

from config import load_config
from data_preprocessing import load_and_split_raw_data, apply_strict_anti_leakage_preprocessing
from graph_builder import build_hetero_graph, graph_level_undersample
from samplers import get_neighbor_loader
from model import HeteroGraphSAGE, manual_weighted_bce, manual_unweighted_bce

def extract_features(model, loader, device, total_samples, orig_feat_dim, emb_dim):
    """
    Mengekstrak embedding GNN dan menggabungkannya dengan fitur tabular transaksi yang bersih.
    Menggunakan pre-alokasi memori numpy untuk efisiensi RAM maksimal.
    """
    model.eval()
    final_features = np.empty((total_samples, orig_feat_dim + emb_dim), dtype=np.float32)
    all_targets = np.empty((total_samples,), dtype=np.int64)
    
    start_time = time.time()
    current_idx = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            
            embeddings = model(
                batch.x_dict, 
                batch.edge_index_dict, 
                num_nodes_dict=num_nodes_dict, 
                return_embeddings=True
            )
            
            batch_size = batch['transaction'].batch_size
            next_idx = min(current_idx + batch_size, total_samples)
            actual_count = next_idx - current_idx
            
            final_features[current_idx:next_idx, :orig_feat_dim] = batch['transaction'].x[:actual_count].cpu().numpy()
            final_features[current_idx:next_idx, orig_feat_dim:] = embeddings[:actual_count].cpu().numpy()
            all_targets[current_idx:next_idx] = batch['transaction'].y[:actual_count].cpu().numpy()
            
            current_idx = next_idx
            
    inference_time = time.time() - start_time
    
    if current_idx < total_samples:
        final_features = final_features[:current_idx]
        all_targets = all_targets[:current_idx]
        
    return final_features, all_targets, inference_time

def find_optimal_threshold(val_probs, val_labels, thresholds=None, min_precision=0.10):
    """
    Mencari threshold optimal tau* yang memaksimalkan F1-score pada Validation set,
    dengan konstrain presisi minimum yang masuk akal secara operasional perbankan.
    """
    if thresholds is None:
        thresholds = np.arange(0.02, 0.70, 0.005)
        
    best_tau = 0.5
    best_f1 = 0.0
    best_metrics = {}
    
    for tau in thresholds:
        preds = (val_probs >= tau).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        prec = precision_score(val_labels, preds, zero_division=0)
        rec = recall_score(val_labels, preds, zero_division=0)
        
        # Saring kandidat dengan batasan presisi minimum
        if f1 > best_f1 and prec >= min_precision:
            best_f1 = f1
            best_tau = float(tau)
            best_metrics = {
                "tau": float(tau),
                "val_f1": float(f1),
                "val_precision": float(prec),
                "val_recall": float(rec)
            }
            
    # Fallback jika konstrain terlalu ketat
    if best_f1 == 0.0:
        for tau in thresholds:
            preds = (val_probs >= tau).astype(int)
            f1 = f1_score(val_labels, preds, zero_division=0)
            prec = precision_score(val_labels, preds, zero_division=0)
            rec = recall_score(val_labels, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_tau = float(tau)
                best_metrics = {
                    "tau": float(tau),
                    "val_f1": float(f1),
                    "val_precision": float(prec),
                    "val_recall": float(rec)
                }

    if best_f1 == 0.0:
        best_tau = 0.5
        best_metrics = {
            "tau": 0.5,
            "val_f1": 0.0,
            "val_precision": 0.0,
            "val_recall": 0.0
        }
        
    return best_tau, best_metrics

def evaluate_gnn(model, loader, device, pos_weight):
    """
    Mengevaluasi representasi GNN pada validation loader.
    """
    model.eval()
    total_loss = 0.0
    total_loss_unw = 0.0
    total_samples = 0
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            
            loss = manual_weighted_bce(probs, targets, pos_weight)
            loss_unw = manual_unweighted_bce(probs, targets)
            
            total_loss += loss.item() * targets.size(0)
            total_loss_unw += loss_unw.item() * targets.size(0)
            total_samples += targets.size(0)
            
            all_probs.append(probs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            
    val_loss = total_loss / total_samples if total_samples > 0 else 0.0
    val_loss_unw = total_loss_unw / total_samples if total_samples > 0 else 0.0
    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)
    
    auprc = float(average_precision_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    roc_auc = float(roc_auc_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    
    return val_loss, val_loss_unw, auprc, roc_auc

def fit_xgb_model(params, train_features, train_labels, val_features, val_labels, use_gpu=True):
    """
    Helper untuk melatih model XGBoost dengan dukungan GPU/CPU fallback.
    """
    eval_set = [(val_features, val_labels)]
    xgb_params = dict(params)
    xgb_params['eval_metric'] = 'aucpr'
    xgb_params['random_state'] = 42
    
    if use_gpu and torch.cuda.is_available():
        try:
            model = XGBClassifier(**xgb_params, tree_method='hist', device='cuda')
            model.fit(train_features, train_labels, eval_set=eval_set, verbose=False)
            return model
        except Exception:
            pass
            
    model = XGBClassifier(**xgb_params, tree_method='hist', n_jobs=-1)
    model.fit(train_features, train_labels, eval_set=eval_set, verbose=False)
    return model

def calculate_gain_distribution(model, feature_names):
    """
    Menghitung split gain untuk fitur Tabular vs 128-d GraphSAGE.
    """
    booster = model.get_booster()
    score_gain = booster.get_score(importance_type='gain')
    score_weight = booster.get_score(importance_type='weight')
    
    total_gain_all = 0.0
    total_gain_tab = 0.0
    total_gain_gnn = 0.0
    importance_list = []
    
    for idx, name in enumerate(feature_names):
        f_key = f"f{idx}"
        gain = float(score_gain.get(f_key, 0.0))
        weight = int(score_weight.get(f_key, 0))
        is_gnn = name.startswith("gnn_emb_")
        total_gain_all += gain
        if is_gnn:
            total_gain_gnn += gain
        else:
            total_gain_tab += gain
            
        importance_list.append({
            "feature_index": idx,
            "feature_name": name,
            "is_gnn": is_gnn,
            "gain": gain,
            "weight": weight
        })
        
    importance_list.sort(key=lambda x: x['gain'], reverse=True)
    pct_tab = (total_gain_tab / max(1e-5, total_gain_all)) * 100
    pct_gnn = (total_gain_gnn / max(1e-5, total_gain_all)) * 100
    
    return {
        "total_gain_all": total_gain_all,
        "total_gain_tabular": total_gain_tab,
        "total_gain_gnn": total_gain_gnn,
        "pct_tabular_gain": pct_tab,
        "pct_gnn_gain": pct_gnn,
        "top_features": importance_list[:30]
    }

def main():
    parser = argparse.ArgumentParser(description="Eksperimen 21B (EXP21B): Balanced Hyperparameter Tuning & Regularization")
    parser.add_argument("--config", type=str, default="config.reference.yaml", help="Path ke file konfigurasi YAML")
    parser.add_argument("--sampling", type=str, default=None, choices=["uniform", "topology", "importance"], help="Metode neighbor sampling")
    parser.add_argument("--nrows", type=int, default=None, help="Batas baris dataset untuk quick run/smoke test")
    parser.add_argument("--tuning-mode", type=str, default=None, choices=["grid", "ablation", "fast"], help="Mode tuning hyperparameter")
    parser.add_argument("--selection-metric", type=str, default=None, choices=["val_f1", "val_auprc", "val_gmean"], help="Metrik pemilihan parameter terbaik")
    parser.add_argument("--epochs", type=int, default=None, help="Override jumlah epoch GNN")
    parser.add_argument("--skip-gnn-train", action="store_true", help="Lewati training GNN jika model checkpoint sudah ada")
    args = parser.parse_args()

    # 1. Pemuatan Konfigurasi
    cfg = load_config(args.config)
    sampling_method = args.sampling if args.sampling is not None else cfg.get('sampling', {}).get('method', 'uniform')
    tuning_cfg = cfg.get('tuning', {})
    tuning_mode = args.tuning_mode if args.tuning_mode is not None else tuning_cfg.get('mode', 'grid')
    selection_metric = args.selection_metric if args.selection_metric is not None else tuning_cfg.get('selection_metric', 'val_f1')
    
    data_cfg = cfg.get('data', {})
    tw_cfg = cfg.get('time_window', {})
    feat_cfg = cfg.get('features', {})
    model_cfg = cfg.get('model', {})
    train_cfg = cfg.get('training', {})
    cls_cfg = cfg.get('classifier', {})
    out_cfg = cfg.get('output', {})

    model_dir = Path(out_cfg.get('model_dir', 'model/exp21b'))
    result_dir = Path(out_cfg.get('result_dir', 'result/exp21b'))
    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 85)
    print("  EKSPERIMEN 21B (EXP21B): BALANCED HYPERPARAMETER TUNING & REGULARIZATION")
    print("  Pipeline Anti-Leakage & Systematic Classifier Parameter Optimization")
    print("=" * 85)
    print(f"Sampling Method      : {sampling_method.upper()}")
    print(f"Tuning Mode          : {tuning_mode.upper()} (Selection Metric: {selection_metric})")
    print(f"Model Directory      : {model_dir}")
    print(f"Result Directory     : {result_dir}")
    print("=" * 85)

    # 2. Phase 0: Strict Chronological Splitting & Out-of-Sample Preprocessing
    print("\n--- [Phase 0: Chronological Splitting & Strict Out-of-Sample Preprocessing] ---")
    data_prep_start = time.time()
    
    file_path = data_cfg.get('transactions')
    start_year = data_cfg.get('start_year', 2016)
    nrows = args.nrows if args.nrows is not None else data_cfg.get('max_rows')
    
    reference_date = tw_cfg.get('reference_date', '2018-01-01')
    train_days = tw_cfg.get('train_days', 300)
    val_days = tw_cfg.get('val_days', 30)
    gap_days = tw_cfg.get('gap_days', 8)
    test_days = tw_cfg.get('test_days', 60)
    
    train_df, val_df, test_df, gap_df, stats_dict = load_and_split_raw_data(
        file_path=file_path,
        start_year=start_year,
        nrows=nrows,
        reference_date=reference_date,
        train_days=train_days,
        val_days=val_days,
        gap_days=gap_days,
        test_days=test_days
    )
    
    stats_path = result_dir / "window_statistics.json"
    with open(stats_path, 'w') as f:
        json.dump(stats_dict, f, indent=4)
    print(f"Statistik jendela waktu tersimpan ke: {stats_path}")

    feature_mode = feat_cfg.get('mode', 'clean')
    use_cyclical = feat_cfg.get('use_cyclical_time', True)
    
    train_df, val_df, test_df, gap_df, feature_cols, prep_artifacts = apply_strict_anti_leakage_preprocessing(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        gap_df=gap_df,
        feature_mode=feature_mode,
        use_cyclical_time=use_cyclical
    )
    
    data_prep_time = time.time() - data_prep_start
    print(f"Preprocessing selesai dalam {data_prep_time:.2f} detik.")

    # 3. Phase 1: Membangun Graf Heterogen & Pelatihan GraphSAGE Encoder
    print("\n--- [Phase 1: Membangun Graf Heterogen & Pelatihan GraphSAGE Encoder] ---")
    train_graph, entity_scalers = build_hetero_graph(train_df, feature_cols, scalers_dict=None)
    val_graph, _ = build_hetero_graph(val_df, feature_cols, scalers_dict=entity_scalers)
    test_graph, _ = build_hetero_graph(test_df, feature_cols, scalers_dict=entity_scalers)

    num_neighbors = train_cfg.get('num_neighbors', [15, 10])
    batch_size = train_cfg.get('batch_size', 1024)
    
    train_loader = get_neighbor_loader(train_graph, num_neighbors, batch_size, shuffle=True, sampling_method=sampling_method)
    val_loader = get_neighbor_loader(val_graph, num_neighbors, batch_size, shuffle=False, sampling_method=sampling_method)
    test_loader = get_neighbor_loader(test_graph, num_neighbors, batch_size, shuffle=False, sampling_method=sampling_method)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device komputasi: {device}")

    feature_dims = {
        'transaction': len(feature_cols),
        'user': train_graph['user'].x.size(1),
        'merchant': train_graph['merchant'].x.size(1)
    }

    hidden_channels = model_cfg.get('hidden_channels', 128)
    out_channels = model_cfg.get('out_channels', 1)
    num_layers = model_cfg.get('num_layers', 2)
    aggr = model_cfg.get('aggr', 'mean')
    dropout = model_cfg.get('dropout', 0.2)

    gnn_model = HeteroGraphSAGE(
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        num_layers=num_layers,
        data_metadata=train_graph.metadata(),
        feature_dims=feature_dims,
        aggr=aggr,
        dropout=dropout
    ).to(device)

    # Bobot loss GNN (Teredam Sqrt)
    num_neg = (train_graph['transaction'].y == 0).sum().item()
    num_pos = (train_graph['transaction'].y == 1).sum().item()
    raw_pos_weight = float(num_neg / max(1, num_pos))
    sqrt_pos_weight = float(np.sqrt(raw_pos_weight))
    
    loss_weight_strat = train_cfg.get('loss_weight_strategy', 'sqrt')
    if loss_weight_strat == 'sqrt':
        gnn_pos_weight = min(sqrt_pos_weight, train_cfg.get('max_pos_weight', 50.0))
    elif loss_weight_strat == 'full':
        gnn_pos_weight = raw_pos_weight
    else:
        gnn_pos_weight = min(float(train_cfg.get('fixed_pos_weight', 27.0)), train_cfg.get('max_pos_weight', 50.0))
        
    print(f"GNN Pos Weight terhitung: {gnn_pos_weight:.2f} (Rasio mentah kelas mayoritas/minoritas: {raw_pos_weight:.2f})")

    lr = train_cfg.get('learning_rate', 0.001)
    weight_decay = train_cfg.get('weight_decay', 0.0001)
    optimizer = optim.Adam(gnn_model.parameters(), lr=lr, weight_decay=weight_decay)
    
    epochs = args.epochs if args.epochs is not None else train_cfg.get('epochs', 20)
    patience = train_cfg.get('early_stopping_patience', 7)
    
    best_gnn_path = model_dir / f"exp21b_{sampling_method}_best_gnn.pt"
    gnn_train_time = 0.0
    best_epoch = 0
    best_val_auprc = 0.0
    gnn_epoch_history = []

    if args.skip_gnn_train and best_gnn_path.exists():
        print(f"\n[INFO] Melompati pelatihan GNN. Memuat bobot checkpoint tersimpan: {best_gnn_path}")
        gnn_model.load_state_dict(torch.load(best_gnn_path, map_location=device))
    else:
        print(f"\nMemulai Pelatihan HeteroGraphSAGE ({epochs} epoch, Patience: {patience})...")
        gnn_train_start = time.time()
        no_improve_epochs = 0
        
        for epoch in range(1, epochs + 1):
            gnn_model.train()
            total_train_loss = 0.0
            total_tx_nodes = 0
            
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
                probs = gnn_model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
                
                batch_size_current = batch['transaction'].batch_size
                probs = probs[:batch_size_current]
                targets = batch['transaction'].y[:batch_size_current]
                
                loss = manual_weighted_bce(probs, targets, gnn_pos_weight)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(gnn_model.parameters(), max_norm=train_cfg.get('grad_clip_norm', 5.0))
                optimizer.step()
                
                total_train_loss += loss.item() * targets.size(0)
                total_tx_nodes += targets.size(0)
                
            train_loss = total_train_loss / total_tx_nodes if total_tx_nodes > 0 else 0.0
            val_loss, val_loss_unw, val_auprc, val_roc = evaluate_gnn(gnn_model, val_loader, device, gnn_pos_weight)
            
            gnn_epoch_history.append({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_auprc": val_auprc,
                "val_roc_auc": val_roc
            })
            
            print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUPRC: {val_auprc:.4f} | Val ROC: {val_roc:.4f}")
            
            if val_auprc > best_val_auprc:
                best_val_auprc = val_auprc
                best_epoch = epoch
                torch.save(gnn_model.state_dict(), best_gnn_path)
                no_improve_epochs = 0
            else:
                no_improve_epochs += 1
                if no_improve_epochs >= patience:
                    print(f"Early stopping terpicu pada epoch {epoch}. Model terbaik dari epoch {best_epoch} (Val AUPRC: {best_val_auprc:.4f}).")
                    break
                    
        gnn_train_time = time.time() - gnn_train_start
        print(f"Pelatihan GNN selesai dalam {gnn_train_time:.2f} detik. Memuat bobot terbaik...")
        gnn_model.load_state_dict(torch.load(best_gnn_path, map_location=device))

    # 4. Phase 2: Ekstraksi Representasi Induktif (Dilakukan Sekali untuk Seluruh Grid)
    print("\n--- [Phase 2: Ekstraksi Representasi Induktif GNN + Tabular] ---")
    orig_dim = len(feature_cols)
    emb_dim = hidden_channels
    
    train_features, train_labels, inf_time_train = extract_features(gnn_model, train_loader, device, len(train_df), orig_dim, emb_dim)
    val_features, val_labels, inf_time_val = extract_features(gnn_model, val_loader, device, len(val_df), orig_dim, emb_dim)
    test_features, test_labels, inf_time_test_gnn = extract_features(gnn_model, test_loader, device, len(test_df), orig_dim, emb_dim)
    
    print(f"Fitur Train terekstraksi: {train_features.shape}")
    print(f"Fitur Val terekstraksi  : {val_features.shape}")
    print(f"Fitur Test terekstraksi : {test_features.shape}")
    
    # Bebaskan memori GPU & PyG
    del train_graph, val_graph, test_graph, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    feature_names = list(feature_cols) + [f"gnn_emb_{i}" for i in range(emb_dim)]

    # 5. Phase 3: Systematic Hyperparameter Tuning (EXP21B Core)
    print("\n" + "="*85)
    print("  PHASE 3: SYSTEMATIC BALANCED HYPERPARAMETER TUNING (EXP21B)")
    print("="*85)

    grid_cfg = tuning_cfg.get('grid', {})
    base_params = tuning_cfg.get('base_params', {
        'n_estimators': 100,
        'learning_rate': 0.05,
        'subsample': 0.8
    })
    min_prec_constraint = tuning_cfg.get('min_precision_constraint', 0.10)

    # Menyiapkan kandidat parameter
    spw_candidates = grid_cfg.get('scale_pos_weight', [1.0, 10.0, 27.0, 50.0])
    depth_candidates = grid_cfg.get('max_depth', [3, 4, 5])
    mcw_candidates = grid_cfg.get('min_child_weight', [5, 20, 50])
    colsample_candidates = grid_cfg.get('colsample_bytree', [0.6, 0.8, 1.0])

    # Konversi string atau float pada scale_pos_weight
    resolved_spw = []
    for val in spw_candidates:
        if isinstance(val, str):
            if val.lower() == 'sqrt':
                resolved_spw.append(float(round(sqrt_pos_weight, 2)))
            elif val.lower() == 'full':
                resolved_spw.append(float(round(raw_pos_weight, 2)))
            else:
                resolved_spw.append(float(val))
        else:
            resolved_spw.append(float(val))
    resolved_spw = sorted(list(set(resolved_spw)))

    # Membangun kombinasi berdasarkan mode tuning
    trials_to_run = []
    if tuning_mode == "grid":
        for spw, depth, mcw, colsample in itertools.product(resolved_spw, depth_candidates, mcw_candidates, colsample_candidates):
            trials_to_run.append({
                "scale_pos_weight": spw,
                "max_depth": depth,
                "min_child_weight": mcw,
                "colsample_bytree": colsample
            })
    elif tuning_mode == "ablation":
        # Anchor baseline: spw=27.0 (sqrt), depth=4, mcw=20, colsample=0.8
        base_anchor = {"scale_pos_weight": float(round(sqrt_pos_weight, 2)), "max_depth": 4, "min_child_weight": 20, "colsample_bytree": 0.8}
        trials_to_run.append(base_anchor)
        
        # Variasi SPW
        for spw in resolved_spw + [float(round(raw_pos_weight, 2))]:
            if spw != base_anchor["scale_pos_weight"]:
                p = dict(base_anchor)
                p["scale_pos_weight"] = spw
                trials_to_run.append(p)
        # Variasi Depth
        for depth in depth_candidates:
            if depth != base_anchor["max_depth"]:
                p = dict(base_anchor)
                p["max_depth"] = depth
                trials_to_run.append(p)
        # Variasi Min Child Weight
        for mcw in mcw_candidates:
            if mcw != base_anchor["min_child_weight"]:
                p = dict(base_anchor)
                p["min_child_weight"] = mcw
                trials_to_run.append(p)
        # Variasi Colsample
        for colsample in colsample_candidates:
            if colsample != base_anchor["colsample_bytree"]:
                p = dict(base_anchor)
                p["colsample_bytree"] = colsample
                trials_to_run.append(p)
    elif tuning_mode == "fast":
        # Focused subset (6 kombinasi cepat untuk verifikasi)
        fast_spw = [1.0, float(round(sqrt_pos_weight, 2)), 50.0]
        fast_depth = [3, 4]
        for spw, depth in itertools.product(fast_spw, fast_depth):
            trials_to_run.append({
                "scale_pos_weight": spw,
                "max_depth": depth,
                "min_child_weight": 20,
                "colsample_bytree": 0.8
            })

    print(f"Total konfigurasi hiperparameter yang akan dieksplorasi: {len(trials_to_run)} kombinasi.")
    print("Mengeksekusi tuning pada Validation Set (Hold-out Chronologic 30 hari)...")
    
    tuning_records = []
    tuning_start_time = time.time()

    for idx, params in enumerate(trials_to_run, 1):
        fit_params = dict(base_params)
        fit_params.update(params)
        
        t0_fit = time.time()
        candidate_model = fit_xgb_model(fit_params, train_features, train_labels, val_features, val_labels)
        fit_time = time.time() - t0_fit
        
        # Prediksi pada Validation set
        val_probs_cand = candidate_model.predict_proba(val_features)[:, 1]
        
        # Evaluasi metrik Validation
        val_auprc_cand = float(average_precision_score(val_labels, val_probs_cand)) if len(np.unique(val_labels)) > 1 else 0.0
        val_roc_cand = float(roc_auc_score(val_labels, val_probs_cand)) if len(np.unique(val_labels)) > 1 else 0.0
        
        # Kalibrasi threshold dinamis tau* pada Validation
        best_tau_cand, tau_stats = find_optimal_threshold(val_probs_cand, val_labels, min_precision=min_prec_constraint)
        
        val_preds_tau = (val_probs_cand >= best_tau_cand).astype(int)
        val_f1_cand = float(f1_score(val_labels, val_preds_tau, zero_division=0))
        val_rec_cand = float(recall_score(val_labels, val_preds_tau, zero_division=0))
        val_prec_cand = float(precision_score(val_labels, val_preds_tau, zero_division=0))
        
        cm_val = confusion_matrix(val_labels, val_preds_tau)
        if cm_val.shape == (2, 2):
            tn_v, fp_v, fn_v, tp_v = [int(v) for v in cm_val.ravel()]
            spec_v = tn_v / (tn_v + fp_v) if (tn_v + fp_v) > 0 else 0.0
            gmean_cand = float(np.sqrt(val_rec_cand * spec_v))
        else:
            gmean_cand = 0.0

        # Distribusi gain Tabular vs GNN
        gain_info = calculate_gain_distribution(candidate_model, feature_names)

        record = {
            "trial_id": idx,
            "scale_pos_weight": float(params['scale_pos_weight']),
            "max_depth": int(params['max_depth']),
            "min_child_weight": int(params['min_child_weight']),
            "colsample_bytree": float(params['colsample_bytree']),
            "val_auprc": val_auprc_cand,
            "val_roc_auc": val_roc_cand,
            "optimal_tau": float(best_tau_cand),
            "val_f1": val_f1_cand,
            "val_recall": val_rec_cand,
            "val_precision": val_prec_cand,
            "val_gmean": gmean_cand,
            "pct_tabular_gain": float(gain_info['pct_tabular_gain']),
            "pct_gnn_gain": float(gain_info['pct_gnn_gain']),
            "total_gain": float(gain_info['total_gain_all']),
            "fit_time_seconds": float(fit_time)
        }
        tuning_records.append(record)

        if idx % 10 == 0 or idx == len(trials_to_run) or idx == 1:
            print(f"[{idx:3d}/{len(trials_to_run):3d}] SPW={params['scale_pos_weight']:5.1f} | Depth={params['max_depth']} | MCW={params['min_child_weight']:2d} | Col={params['colsample_bytree']:.1f} --> Val F1: {val_f1_cand:.4f} (Rec: {val_rec_cand*100:.1f}%, Prec: {val_prec_cand*100:.1f}%) | Val AUPRC: {val_auprc_cand:.4f} | GNN Gain: {gain_info['pct_gnn_gain']:4.1f}%")

    total_tuning_time = time.time() - tuning_start_time
    print(f"\nSelesai mengeksplorasi {len(trials_to_run)} kombinasi dalam {total_tuning_time:.2f} detik.")

    # Simpan hasil tuning lengkap ke JSON dan CSV
    tuning_df = pd.DataFrame(tuning_records)
    csv_path = result_dir / f"exp21b_{sampling_method}_tuning_results.csv"
    tuning_df.to_csv(csv_path, index=False)
    
    json_path = result_dir / f"exp21b_{sampling_method}_tuning_results.json"
    with open(json_path, 'w') as f:
        json.dump(tuning_records, f, indent=4)
    print(f"Log lengkap tuning tersimpan di:\n  • {csv_path}\n  • {json_path}")

    # Pilih Model Terbaik berdasarkan Selection Metric
    # Prioritaskan metrik yang dipilih, jika seri prioritaskan val_auprc
    tuning_records_sorted = sorted(
        tuning_records, 
        key=lambda x: (x.get(selection_metric, 0.0), x.get('val_auprc', 0.0), x.get('val_recall', 0.0)), 
        reverse=True
    )
    best_candidate = tuning_records_sorted[0]

    print("\n" + "="*85)
    print(f"  KONFIGURASI TERBAIK BERDASARKAN {selection_metric.upper()} PADA VALIDATION SET")
    print("="*85)
    print(f"  • Trial ID          : #{best_candidate['trial_id']}")
    print(f"  • scale_pos_weight  : {best_candidate['scale_pos_weight']}")
    print(f"  • max_depth         : {best_candidate['max_depth']}")
    print(f"  • min_child_weight  : {best_candidate['min_child_weight']}")
    print(f"  • colsample_bytree  : {best_candidate['colsample_bytree']}")
    print(f"  • Optimal tau* (Val): {best_candidate['optimal_tau']:.3f}")
    print(f"  • Validation F1     : {best_candidate['val_f1']:.4f}")
    print(f"  • Validation Recall : {best_candidate['val_recall']*100:.2f}%")
    print(f"  • Validation Prec   : {best_candidate['val_precision']*100:.2f}%")
    print(f"  • Validation AUPRC  : {best_candidate['val_auprc']:.4f}")
    print(f"  • GNN Gain Share    : {best_candidate['pct_gnn_gain']:.2f}% (Tabular: {best_candidate['pct_tabular_gain']:.2f}%)")
    print("="*85)

    # 6. Phase 4: Pelatihan Model Terbaik & Evaluasi Final pada Test Set Masa Depan (60 Hari)
    print("\n--- [Phase 4: Pelatihan Model Terbaik & Evaluasi Final Test Set] ---")
    best_params = dict(base_params)
    best_params.update({
        'scale_pos_weight': best_candidate['scale_pos_weight'],
        'max_depth': best_candidate['max_depth'],
        'min_child_weight': best_candidate['min_child_weight'],
        'colsample_bytree': best_candidate['colsample_bytree']
    })

    t0_best_fit = time.time()
    best_xgb_model = fit_xgb_model(best_params, train_features, train_labels, val_features, val_labels)
    best_fit_time = time.time() - t0_best_fit

    # Simpan model XGBoost terbaik
    best_model_path = model_dir / f"exp21b_{sampling_method}_best_xgb.json"
    best_xgb_model.save_model(best_model_path)
    print(f"Model XGBoost terbaik tersimpan di: {best_model_path}")

    # Evaluasi Prediksi pada Validation & Test Set
    val_probs_best = best_xgb_model.predict_proba(val_features)[:, 1]
    final_tau, final_val_stats = find_optimal_threshold(val_probs_best, val_labels, min_precision=min_prec_constraint)
    
    t0_test_eval = time.time()
    test_probs_best = best_xgb_model.predict_proba(test_features)[:, 1]
    test_inf_time = time.time() - t0_test_eval
    total_test_inf_time = inf_time_test_gnn + test_inf_time

    # Evaluasi Baseline tau = 0.50
    preds_05 = (test_probs_best >= 0.50).astype(int)
    f1_05 = float(f1_score(test_labels, preds_05, zero_division=0))
    rec_05 = float(recall_score(test_labels, preds_05, zero_division=0))
    prec_05 = float(precision_score(test_labels, preds_05, zero_division=0))

    # Evaluasi Optimal tau*
    preds_tau = (test_probs_best >= final_tau).astype(int)
    f1_tau = float(f1_score(test_labels, preds_tau, zero_division=0))
    f1_macro_tau = float(f1_score(test_labels, preds_tau, average='macro', zero_division=0))
    rec_tau = float(recall_score(test_labels, preds_tau, zero_division=0))
    prec_tau = float(precision_score(test_labels, preds_tau, zero_division=0))
    acc_tau = float(accuracy_score(test_labels, preds_tau))

    cm = confusion_matrix(test_labels, preds_tau)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = [int(v) for v in cm.ravel()]
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        gmean = float(np.sqrt(rec_tau * specificity))
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
        specificity, gmean = 0.0, 0.0

    roc_auc = float(roc_auc_score(test_labels, test_probs_best)) if len(np.unique(test_labels)) > 1 else 0.0
    auprc = float(average_precision_score(test_labels, test_probs_best)) if len(np.unique(test_labels)) > 1 else 0.0

    # Metrik Operasional Perbankan
    top_1pct_count = max(1, int(round(len(test_labels) * 0.01)))
    top_1pct_idx = np.argsort(test_probs_best)[::-1][:top_1pct_count]
    precision_1pct = float(test_labels[top_1pct_idx].mean())
    base_fraud_rate = float(test_labels.mean())
    lift_1pct = float(precision_1pct / base_fraud_rate) if base_fraud_rate > 0 else 0.0

    top_500_idx = np.argsort(test_probs_best)[::-1][:500]
    tp_at_500 = int(test_labels[top_500_idx].sum())
    precision_at_500 = float(tp_at_500 / 500)

    # Audit Feature Importance Terbaik
    best_gain_info = calculate_gain_distribution(best_xgb_model, feature_names)

    print("\n" + "="*85)
    print(f"  HASIL EVALUASI AKHIR MODEL TERBAIK PADA TEST SET MASA DEPAN (60 HARI, N={len(test_labels):,})")
    print("="*85)
    print(f"  • Metrik Bebas Threshold (Threshold-Independent):")
    print(f"      - AUCPR (Area Under PR Curve) : {auprc:.4f}")
    print(f"      - ROC-AUC                     : {roc_auc:.4f}")
    print(f"  • Metrik Operasional Perbankan:")
    print(f"      - Lift @ 1% Transaksi Teratas : {lift_1pct:.2f}x")
    print(f"      - TP @ 500 Alert Harian       : {tp_at_500} / 500 ({precision_at_500*100:.1f}%)")
    print(f"  • Metrik pada Threshold Optimal tau* = {final_tau:.3f}:")
    print(f"      - F1-Score                    : {f1_tau:.4f}")
    print(f"      - Recall                      : {rec_tau*100:.2f}% (TP={tp}, FN={fn})")
    print(f"      - Precision                   : {prec_tau*100:.2f}% (FP={fp})")
    print(f"      - Akurasi                     : {acc_tau*100:.4f}%")
    print(f"      - G-Mean                      : {gmean:.4f}")
    print(f"  • Metrik pada Baseline Threshold 0.50:")
    print(f"      - F1-Score: {f1_05:.4f} | Recall: {rec_05*100:.2f}% | Precision: {prec_05*100:.2f}%")
    print(f"  • Feature Split Gain Distribution:")
    print(f"      - Clean Tabular Features      : {best_gain_info['total_gain_tabular']:12.2f} ({best_gain_info['pct_tabular_gain']:.2f}%)")
    print(f"      - GraphSAGE 128-d Embeddings  : {best_gain_info['total_gain_gnn']:12.2f} ({best_gain_info['pct_gnn_gain']:.2f}%)")
    print("="*85)

    # 7. Phase 5: Studi Ablasi Komparatif untuk Bab 4 Skripsi
    print("\n--- [Phase 5: Eksekusi Studi Ablasi Komparatif untuk Bab 4 Skripsi] ---")
    ablation_summary = {
        "scale_pos_weight_ablation": [],
        "feature_subset_ablation": []
    }

    # A. Ablasi SPW: 1.0 vs 10.0 vs 27.0 vs 50.0 vs 732.0 (Full)
    spw_tests = [1.0, 10.0, float(round(sqrt_pos_weight, 2)), 50.0, float(round(raw_pos_weight, 2))]
    print("Menjalankan Ablasi Class-Weight Damping (scale_pos_weight)...")
    for test_spw in spw_tests:
        p_abl = dict(best_params)
        p_abl['scale_pos_weight'] = test_spw
        m_abl = fit_xgb_model(p_abl, train_features, train_labels, val_features, val_labels)
        
        # Evaluasi pada Test
        val_p = m_abl.predict_proba(val_features)[:, 1]
        t_opt, _ = find_optimal_threshold(val_p, val_labels, min_precision=min_prec_constraint)
        test_p = m_abl.predict_proba(test_features)[:, 1]
        
        t_auprc = float(average_precision_score(test_labels, test_p)) if len(np.unique(test_labels)) > 1 else 0.0
        preds_opt = (test_p >= t_opt).astype(int)
        t_f1 = float(f1_score(test_labels, preds_opt, zero_division=0))
        t_rec = float(recall_score(test_labels, preds_opt, zero_division=0))
        t_prec = float(precision_score(test_labels, preds_opt, zero_division=0))
        g_info = calculate_gain_distribution(m_abl, feature_names)
        
        ablation_summary["scale_pos_weight_ablation"].append({
            "scale_pos_weight": test_spw,
            "optimal_tau": t_opt,
            "test_auprc": t_auprc,
            "test_f1": t_f1,
            "test_recall": t_rec,
            "test_precision": t_prec,
            "pct_gnn_gain": g_info['pct_gnn_gain']
        })
        print(f"  • SPW={test_spw:6.1f} | tau*={t_opt:.3f} | Test F1={t_f1:.4f} | Recall={t_rec*100:.1f}% | Prec={t_prec*100:.1f}% | GNN Gain={g_info['pct_gnn_gain']:.1f}%")

    # B. Ablasi Subset Fitur: Tabular Only vs GraphSAGE Only vs Hybrid Tuned
    print("\nMenjalankan Ablasi Subset Fitur (Tabular vs GNN vs Hybrid)...")
    subset_definitions = {
        "Tabular Only": (train_features[:, :orig_dim], val_features[:, :orig_dim], test_features[:, :orig_dim]),
        "GraphSAGE Only": (train_features[:, orig_dim:], val_features[:, orig_dim:], test_features[:, orig_dim:]),
        "Hybrid (Tabular + GNN)": (train_features, val_features, test_features)
    }

    for s_name, (tr_f, vl_f, ts_f) in subset_definitions.items():
        m_sub = fit_xgb_model(best_params, tr_f, train_labels, vl_f, val_labels)
        vl_p = m_sub.predict_proba(vl_f)[:, 1]
        t_opt_sub, _ = find_optimal_threshold(vl_p, val_labels, min_precision=min_prec_constraint)
        ts_p = m_sub.predict_proba(ts_f)[:, 1]
        
        s_auprc = float(average_precision_score(test_labels, ts_p)) if len(np.unique(test_labels)) > 1 else 0.0
        preds_sub = (ts_p >= t_opt_sub).astype(int)
        s_f1 = float(f1_score(test_labels, preds_sub, zero_division=0))
        s_rec = float(recall_score(test_labels, preds_sub, zero_division=0))
        s_prec = float(precision_score(test_labels, preds_sub, zero_division=0))
        
        ablation_summary["feature_subset_ablation"].append({
            "subset_name": s_name,
            "optimal_tau": t_opt_sub,
            "test_auprc": s_auprc,
            "test_f1": s_f1,
            "test_recall": s_rec,
            "test_precision": s_prec
        })
        print(f"  • {s_name:24s} | Test F1={s_f1:.4f} | Recall={s_rec*100:.1f}% | Prec={s_prec*100:.1f}% | AUCPR={s_auprc:.4f}")

    # Simpan hasil ringkasan ablasi
    ablation_path = result_dir / f"exp21b_{sampling_method}_ablation_summary.json"
    with open(ablation_path, 'w') as f:
        json.dump(ablation_summary, f, indent=4)
    print(f"Hasil ablasi tersimpan di: {ablation_path}")

    # 8. Simpan Berkas Artefak Hasil Lengkap
    # A. Feature Importance Audit
    imp_path = result_dir / f"exp21b_{sampling_method}_feature_importance.json"
    with open(imp_path, 'w') as f:
        json.dump(best_gain_info, f, indent=4)

    # B. Hasil Evaluasi Model Utama
    results = {
        "experiment_id": "EXP21B",
        "experiment_name": "Balanced Hyperparameter Tuning & Anti-Memorization Regularization",
        "sampling_method": sampling_method,
        "tuning_mode": tuning_mode,
        "selection_metric": selection_metric,
        "best_hyperparameters": best_params,
        "time_window": {
            "reference_date": reference_date,
            "train_days": train_days,
            "val_days": val_days,
            "gap_days": gap_days,
            "test_days": test_days
        },
        "threshold_calibration": {
            "optimal_tau": float(final_tau),
            "val_f1": float(final_val_stats.get('val_f1', 0.0)),
            "val_precision": float(final_val_stats.get('val_precision', 0.0)),
            "val_recall": float(final_val_stats.get('val_recall', 0.0))
        },
        "metrics_optimal_threshold": {
            "threshold": float(final_tau),
            "f1_score": f1_tau,
            "f1_macro": f1_macro_tau,
            "recall": rec_tau,
            "precision": prec_tau,
            "accuracy": acc_tau,
            "gmean": gmean,
            "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp}
        },
        "metrics_default_threshold_05": {
            "threshold": 0.50,
            "f1_score": f1_05,
            "recall": rec_05,
            "precision": prec_05
        },
        "metrics_threshold_independent": {
            "auprc": auprc,
            "roc_auc": roc_auc,
            "lift_1_percent": lift_1pct,
            "tp_at_500": tp_at_500,
            "precision_at_500": precision_at_500
        },
        "feature_importance_summary": {
            "total_gain_tabular": float(best_gain_info['total_gain_tabular']),
            "total_gain_gnn": float(best_gain_info['total_gain_gnn']),
            "pct_tabular_gain": float(best_gain_info['pct_tabular_gain']),
            "pct_gnn_gain": float(best_gain_info['pct_gnn_gain'])
        },
        "runtime_seconds": {
            "data_preprocessing": data_prep_time,
            "gnn_training": gnn_train_time,
            "tuning_exploration": total_tuning_time,
            "best_xgb_fit": best_fit_time,
            "total_test_inference": total_test_inf_time,
            "total_pipeline": data_prep_time + gnn_train_time + total_tuning_time + best_fit_time + total_test_inf_time
        }
    }

    results_path = result_dir / f"exp21b_{sampling_method}_results.json"
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"\nHasil metrik utama tersimpan di: {results_path}")

    # C. Riwayat Pelatihan GNN
    history_path = result_dir / f"exp21b_{sampling_method}_history.json"
    with open(history_path, 'w') as f:
        json.dump(gnn_epoch_history, f, indent=4)

    # D. Array Prediksi Kontinu untuk Visualisasi
    pred_path = result_dir / f"exp21b_{sampling_method}_predictions.npz"
    np.savez_compressed(
        pred_path,
        test_probs=test_probs_best.astype(np.float32),
        test_labels=test_labels.astype(np.int8),
        val_probs=val_probs_best.astype(np.float32),
        val_labels=val_labels.astype(np.int8)
    )
    print(f"Array prediksi tersimpan ke: {pred_path}")
    print("\n>>> Pipeline EXP21B Selesai dengan Sukses! <<<")

if __name__ == "__main__":
    main()



In [ ]:
%%writefile run_all.py
import os
import subprocess
import sys
import json
import yaml
from pathlib import Path

def run_script(script_name, args=[]):
    print(f"\n{'='*75}")
    print(f"  >>> MENJALANKAN SCRIPT: {script_name} {' '.join(args)}")
    print(f"{'='*75}\n")
    
    command = [sys.executable, script_name] + args
    result = subprocess.run(command)
    
    if result.returncode != 0:
        print(f"\n[ERROR] Eksekusi {script_name} gagal dengan kode {result.returncode}!")
        sys.exit(result.returncode)
    else:
        print(f"\n[SUCCESS] {script_name} selesai dieksekusi.\n")

if __name__ == "__main__":
    print("="*75)
    print("  MEMULAI RUN ALL: EKSPERIMEN 21B (BALANCED HYPERPARAMETER TUNING)")
    print("="*75)
    
    # 1. Install dependencies jika di environment Kaggle/Colab
    if os.path.exists("/kaggle/input") or os.path.exists("/content"):
        print("\n[INFO] Menginstall dependencies (torch_geometric & pyg-lib)...")
        try:
            import torch
            pt_version = torch.__version__.split('+')[0]
            cuda_version = torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
            if cuda_version != 'cpu':
                url = f"https://data.pyg.org/whl/torch-{pt_version}+cu{cuda_version}.html"
                os.system(f"pip install torch_geometric pyg_lib torch_scatter torch_sparse -f {url} > /dev/null 2>&1")
            else:
                os.system("pip install torch_geometric > /dev/null 2>&1")
        except Exception as e:
            print(f"[WARNING] Fallback install torch_geometric: {e}")
            os.system("pip install torch_geometric > /dev/null 2>&1")

    # 2. Deteksi path konfigurasi
    if os.path.exists("/kaggle/input"):
        config_path = "config.kaggle.yaml"
        # Deteksi otomatis CSV di Kaggle
        csv_path = None
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'credit_card_transactions-ibm_v2.csv' in files:
                csv_path = os.path.join(root, 'credit_card_transactions-ibm_v2.csv')
                break
        if csv_path:
            print(f"[INFO] Dataset ditemukan di: {csv_path}")
            with open(config_path, 'r') as f:
                cfg = yaml.safe_load(f)
            cfg['data']['transactions'] = csv_path
            with open(config_path, 'w') as f:
                yaml.dump(cfg, f)
    elif os.path.exists("/content"):
        config_path = "config.colab.yaml"
    else:
        config_path = "config.reference.yaml"

    with open(config_path, 'r') as f:
        cfg_loaded = yaml.safe_load(f)
    res_dir = Path(cfg_loaded.get('output', {}).get('result_dir', 'result/exp21b'))
    res_dir.mkdir(parents=True, exist_ok=True)

    # 3. Jalankan eksperimen untuk metode sampling
    sampling_methods = ["uniform", "topology", "importance"]
    pass_through_args = sys.argv[1:]
    
    for method in sampling_methods:
        print(f"\n{'*'*75}")
        print(f"  >>> MULAI EXP21B DENGAN SAMPLING: {method.upper()}")
        print(f"{'*'*75}\n")
        cmd_args = ["--config", config_path, "--sampling", method] + pass_through_args
        run_script("main.py", cmd_args)
        
    # 4. Generate Visualisasi Hasil (Chart Ilmiah 300 DPI)
    if os.path.exists("visualize_results.py"):
        print("\n" + "="*75)
        print("  >>> MENJALANKAN GENERASI VISUALISASI HASIL EXP21B")
        print("="*75 + "\n")
        try:
            plots_dir = res_dir / 'plots'
            run_script("visualize_results.py", ["--result-dir", str(res_dir), "--output-dir", str(plots_dir)])
        except Exception as e:
            print(f"[WARNING] Gagal menjalankan visualisasi otomatis: {e}")

    # 5. Penyusunan Ringkasan Komparatif Markdown (Master Summary Bab 4)
    summary_report_path = res_dir / "exp21b_summary_report.md"
    summary_lines = [
        "# Ringkasan Hasil Eksperimen 21B (EXP21B): Balanced Hyperparameter Tuning\n",
        "Dokumen ini memuat rangkuman hasil evaluasi komparatif model terbaik setelah hyperparameter tuning berimbang pada Test Set masa depan (60 hari).\n\n",
        "| Neighbor Sampler | Best Parameters (SPW, Depth, MCW, Col) | Test F1-Score | Test Recall | Test Precision | Test AUCPR | ROC-AUC | GNN Gain Share (%) |\n",
        "| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |\n"
    ]

    for method in sampling_methods:
        res_file = res_dir / f"exp21b_{method}_results.json"
        if res_file.exists():
            with open(res_file, 'r') as f:
                d = json.load(f)
            bp = d.get('best_hyperparameters', {})
            param_str = f"SPW={bp.get('scale_pos_weight')}, d={bp.get('max_depth')}, mcw={bp.get('min_child_weight')}, col={bp.get('colsample_bytree')}"
            f1 = d.get('metrics_optimal_threshold', {}).get('f1_score', 0.0)
            rec = d.get('metrics_optimal_threshold', {}).get('recall', 0.0) * 100
            prec = d.get('metrics_optimal_threshold', {}).get('precision', 0.0) * 100
            aucpr = d.get('metrics_threshold_independent', {}).get('auprc', 0.0)
            roc = d.get('metrics_threshold_independent', {}).get('roc_auc', 0.0)
            gnn_pct = d.get('feature_importance_summary', {}).get('pct_gnn_gain', 0.0)
            summary_lines.append(f"| **{method.capitalize()}** | `{param_str}` | **{f1:.4f}** | {rec:.2f}% | {prec:.2f}% | **{aucpr:.4f}** | {roc:.4f} | {gnn_pct:.2f}% |\n")

    summary_lines.append("\n\n*Catatan: Evaluasi dilakukan pada data out-of-sample murni tanpa kebocoran Zip/City dengan threshold dinamis optimal tau*.*")
    
    with open(summary_report_path, 'w', encoding='utf-8') as f:
        f.writelines(summary_lines)
    print(f"\n[INFO] Laporan ringkasan komparasi tersimpan di: {summary_report_path}")

    print("\n" + "="*75)
    print("  🎉 SELURUH PROSES PELATIHAN, TUNING, EVALUASI, DAN AUDIT EXP21B SELESAI! 🎉")
    print("="*75)



In [ ]:
!python run_all.py